In [50]:
import os
import urllib.request

DATA_ROOT = "/content/data/Amazon-toys"

os.makedirs(DATA_ROOT, exist_ok=True)

# BERT config
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/mamuyang/MrTransformer/master/data/Toys/bert_config_toys_64_p11.json",
    os.path.join(DATA_ROOT, "bert_config_toys_64_p11.json")
)

# Toys sequence data
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/mamuyang/MrTransformer/master/data/Toys/toys_pro.txt",
    os.path.join(DATA_ROOT, "toys_pro.txt")
)

print("Downloaded successfully.")
print(os.listdir(DATA_ROOT))

Downloaded successfully.
['bert_config_toys_64_p11.json', 'toys_pro.txt']


In [51]:
import os

for filename in os.listdir(DATA_ROOT):
    path = os.path.join(DATA_ROOT, filename)
    print(filename, "->", os.path.getsize(path) / 1024**2, "MB")

bert_config_toys_64_p11.json -> 0.00031757354736328125 MB
toys_pro.txt -> 1.789316177368164 MB


In [52]:
# ============================================================
# CELL 1 — Imports + Seed + Device
# ============================================================

import os
import json
import math
import random
import collections
import numpy as np
# import tensorflow as tf

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import IterableDataset, DataLoader
from tqdm.auto import tqdm


# -------------------------
# Seed
# -------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# -------------------------
# Device
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

Device: cuda
GPU: Tesla T4
VRAM: 14.56 GB


In [53]:
# ============================================================
# CELL 2 — CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

MAX_SEQ_LENGTH = 50

HIDDEN_SIZE = 64
NUM_HIDDEN_LAYERS = 2
NUM_ATTENTION_HEADS = 2
INTERMEDIATE_SIZE = 256

DROPOUT = 0.2

PREFERENCE_SIZE = 11
TYPE_VOCAB_SIZE = 16


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

BATCH_SIZE = 256

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01

GRAD_CLIP = 5.0

EPOCHS_PRETRAIN = 20
EPOCHS_FINETUNE = 20


# ------------------------------------------------------------
# SSL
# ------------------------------------------------------------

SSL_COMMON_WEIGHT = 10.0
SSL_NEW_PRED_WEIGHT = 0.1


# ------------------------------------------------------------
# Coverage
# ------------------------------------------------------------

COVERAGE_WEIGHT = 1.0


# ------------------------------------------------------------
# Warmup
# Original code:
# 6000 / 600000 = 1%
# ------------------------------------------------------------

WARMUP_RATIO = 0.01


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

DATA_ROOT = "/content/data/Amazon-toys"

BERT_CONFIG_FILE = os.path.join(
    DATA_ROOT,
    "bert_config_toys_64_p11.json"
)

TRAIN_FILE = os.path.join(
    DATA_ROOT,
    "toysdefault.train_p11_learning1_faiss.tfrecord"
)

TEST_FILE = os.path.join(
    DATA_ROOT,
    "toysdefault.test_p11.tfrecord"
)

VOCAB_FILE = os.path.join(
    DATA_ROOT,
    "toysdefault.vocab_p11"
)

HISTORY_FILE = os.path.join(
    DATA_ROOT,
    "toysdefault.his_p11"
)


# ------------------------------------------------------------
# Checkpoints
# ------------------------------------------------------------

CHECKPOINT_DIR = "/content/checkpoint/toys_pytorch"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_PRETRAIN_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best_pretrain.pt"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best_model.pt"
)

LAST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "last_model.pt"
)


print("Configuration loaded.")
print("TRAIN_FILE:", TRAIN_FILE)
print("TEST_FILE :", TEST_FILE)
print("VOCAB_FILE:", VOCAB_FILE)
print("CHECKPOINT:", CHECKPOINT_DIR)

Configuration loaded.
TRAIN_FILE: /content/data/Amazon-toys/toysdefault.train_p11_learning1_faiss.tfrecord
TEST_FILE : /content/data/Amazon-toys/toysdefault.test_p11.tfrecord
VOCAB_FILE: /content/data/Amazon-toys/toysdefault.vocab_p11
CHECKPOINT: /content/checkpoint/toys_pytorch


In [54]:
# ============================================================
# CELL 2 — CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

MAX_SEQ_LENGTH = 50

HIDDEN_SIZE = 64
NUM_HIDDEN_LAYERS = 2
NUM_ATTENTION_HEADS = 2
INTERMEDIATE_SIZE = 256

DROPOUT = 0.2

PREFERENCE_SIZE = 11
TYPE_VOCAB_SIZE = 16


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

BATCH_SIZE = 256

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01

GRAD_CLIP = 5.0

EPOCHS_PRETRAIN = 20
EPOCHS_FINETUNE = 20


# ------------------------------------------------------------
# SSL
# ------------------------------------------------------------

SSL_COMMON_WEIGHT = 10.0
SSL_NEW_PRED_WEIGHT = 0.1


# ------------------------------------------------------------
# Coverage
# ------------------------------------------------------------

COVERAGE_WEIGHT = 1.0


# ------------------------------------------------------------
# Warmup
# Original code:
# 6000 / 600000 = 1%
# ------------------------------------------------------------

WARMUP_RATIO = 0.01


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

DATA_ROOT = "/content/data/Amazon-toys"

BERT_CONFIG_FILE = os.path.join(
    DATA_ROOT,
    "bert_config_toys_64_p11.json"
)

TRAIN_FILE = os.path.join(
    DATA_ROOT,
    "toysdefault.train_p11_learning1_faiss.tfrecord"
)

TEST_FILE = os.path.join(
    DATA_ROOT,
    "toysdefault.test_p11.tfrecord"
)

VOCAB_FILE = os.path.join(
    DATA_ROOT,
    "toysdefault.vocab_p11"
)

HISTORY_FILE = os.path.join(
    DATA_ROOT,
    "toysdefault.his_p11"
)


# ------------------------------------------------------------
# Checkpoints
# ------------------------------------------------------------

CHECKPOINT_DIR = "/content/checkpoint/toys_pytorch"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_PRETRAIN_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best_pretrain.pt"
)

BEST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best_model.pt"
)

LAST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "last_model.pt"
)


print("Configuration loaded.")
print("TRAIN_FILE:", TRAIN_FILE)
print("TEST_FILE :", TEST_FILE)
print("VOCAB_FILE:", VOCAB_FILE)
print("CHECKPOINT:", CHECKPOINT_DIR)

Configuration loaded.
TRAIN_FILE: /content/data/Amazon-toys/toysdefault.train_p11_learning1_faiss.tfrecord
TEST_FILE : /content/data/Amazon-toys/toysdefault.test_p11.tfrecord
VOCAB_FILE: /content/data/Amazon-toys/toysdefault.vocab_p11
CHECKPOINT: /content/checkpoint/toys_pytorch


In [55]:
# ============================================================
# CELL 3 — Check Files
# ============================================================

for path in [
    BERT_CONFIG_FILE,
    TRAIN_FILE,
    TEST_FILE,
    VOCAB_FILE,
    HISTORY_FILE
]:
    print(
        "[OK]" if os.path.exists(path) else "[MISSING]",
        path
    )

[OK] /content/data/Amazon-toys/bert_config_toys_64_p11.json
[MISSING] /content/data/Amazon-toys/toysdefault.train_p11_learning1_faiss.tfrecord
[MISSING] /content/data/Amazon-toys/toysdefault.test_p11.tfrecord
[MISSING] /content/data/Amazon-toys/toysdefault.vocab_p11
[MISSING] /content/data/Amazon-toys/toysdefault.his_p11


In [56]:
# ============================================================
# CELL 4 — BERT CONFIG
# ============================================================

class BertConfig:

    def __init__(
        self,
        vocab_size,
        hidden_size=64,
        num_hidden_layers=2,
        num_attention_heads=2,
        intermediate_size=256,
        hidden_dropout_prob=0.2,
        attention_probs_dropout_prob=0.2,
        max_position_embeddings=50,
        type_vocab_size=16,
        preference_size=11
    ):

        self.vocab_size = vocab_size

        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads

        self.intermediate_size = intermediate_size

        self.hidden_dropout_prob = hidden_dropout_prob
        self.attention_probs_dropout_prob = (
            attention_probs_dropout_prob
        )

        self.max_position_embeddings = max_position_embeddings

        self.type_vocab_size = type_vocab_size

        self.preference_size = preference_size


    @classmethod
    def from_json(cls, path):

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        config = cls(
            vocab_size=data.get("vocab_size", 1000)
        )

        for key, value in data.items():
            setattr(config, key, value)

        return config


# Load original configuration
config = BertConfig.from_json(BERT_CONFIG_FILE)


# Runtime values
config.hidden_size = HIDDEN_SIZE
config.num_hidden_layers = NUM_HIDDEN_LAYERS
config.num_attention_heads = NUM_ATTENTION_HEADS
config.intermediate_size = INTERMEDIATE_SIZE

config.hidden_dropout_prob = DROPOUT
config.attention_probs_dropout_prob = DROPOUT

config.max_position_embeddings = MAX_SEQ_LENGTH
config.preference_size = PREFERENCE_SIZE
config.type_vocab_size = TYPE_VOCAB_SIZE


print("BERT configuration loaded.")
print(vars(config))

BERT configuration loaded.
{'vocab_size': 11936, 'hidden_size': 64, 'num_hidden_layers': 2, 'num_attention_heads': 2, 'intermediate_size': 256, 'hidden_dropout_prob': 0.2, 'attention_probs_dropout_prob': 0.2, 'max_position_embeddings': 50, 'type_vocab_size': 16, 'preference_size': 11, 'hidden_act': 'gelu', 'initializer_range': 0.02}


In [57]:
# # ============================================================
# # CELL 5 — LOAD VOCAB
# # ============================================================

# def load_vocab(vocab_file):

#     vocab = collections.OrderedDict()

#     with open(vocab_file, "r", encoding="utf-8") as f:

#         for index, line in enumerate(f):

#             token = line.strip()

#             if token:
#                 vocab[token] = index

#     vocab_size = len(vocab)

#     print("Vocab size:", vocab_size)

#     return vocab, vocab_size


# # vocab, vocab_size = load_vocab(VOCAB_FILE)

# config.vocab_size = vocab_size
# ============================================================
# CELL 5 — VOCAB SIZE FROM BERT CONFIG
# ============================================================

# vocab file نداریم، بنابراین vocab را از فایل جداگانه نمی‌خوانیم.

print("Vocab size from config:", config.vocab_size)

Vocab size from config: 11936


In [58]:
# ============================================================
# CELL — BUILD USER SEQUENCES FROM toys_pro.txt
# ============================================================

import os
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader


TOYS_FILE = "/content/data/Amazon-toys/toys_pro.txt"

MAX_LEN = 50
MIN_INTERACTIONS = 5


# ============================================================
# 1. Read user-item interactions
# ============================================================

user_sequences_raw = {}

with open(
    TOYS_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if not line:
            continue

        parts = line.split()

        if len(parts) < 2:
            continue

        user = int(parts[0])
        item = int(parts[1])

        if user not in user_sequences_raw:
            user_sequences_raw[user] = []

        user_sequences_raw[user].append(item)


# ============================================================
# 2. Remove users with fewer than 5 interactions
# ============================================================

user_sequences_raw = {

    user: seq

    for user, seq in user_sequences_raw.items()

    if len(seq) >= MIN_INTERACTIONS
}


print(
    "Users after filtering:",
    len(user_sequences_raw)
)


lengths = [
    len(seq)
    for seq in user_sequences_raw.values()
]


print(
    "Average sequence length:",
    np.mean(lengths)
)

print(
    "Max sequence length:",
    np.max(lengths)
)

print(
    "Min sequence length:",
    np.min(lengths)
)


# ============================================================
# 3. Create ITEM ID mapping
#
# We reserve:
# 0 = padding
# ============================================================

all_items = sorted({

    item

    for seq in user_sequences_raw.values()

    for item in seq
})


item2id = {
    item: idx + 1
    for idx, item in enumerate(all_items)
}


id2item = {
    idx: item
    for item, idx in item2id.items()
}


NUM_ITEMS = len(item2id)

PAD_ID = 0


print(
    "Number of items:",
    NUM_ITEMS
)


# ============================================================
# 4. Encode sequences
# ============================================================

user_sequences = {

    user: [
        item2id[item]
        for item in seq
    ]

    for user, seq in user_sequences_raw.items()
}


print(
    "\nExample sequences:"
)

for user, seq in list(
    user_sequences.items()
)[:3]:

    print(
        "User:",
        user,
        "->",
        seq[:20]
    )

Users after filtering: 19412
Average sequence length: 8.633680197815783
Max sequence length: 550
Min sequence length: 5
Number of items: 11924

Example sequences:
User: 1 -> [1, 2, 3, 4, 5]
User: 2 -> [4, 5, 6, 7, 8]
User: 3 -> [9, 10, 11, 12, 4, 5, 13, 14, 15]


In [59]:
# ============================================================
# CELL — TRAIN / VALIDATION / TEST SPLIT
# ============================================================

train_sequences = {}

val_targets = {}

test_targets = {}


for user, seq in user_sequences.items():

    # We need at least:
    # history + validation + test

    if len(seq) < 5:
        continue


    # Last item = test
    test_targets[user] = seq[-1]

    # Second last = validation
    val_targets[user] = seq[-2]

    # Everything before those two = training sequence
    train_sequences[user] = seq[:-2]


print(
    "Train users:",
    len(train_sequences)
)

print(
    "Validation users:",
    len(val_targets)
)

print(
    "Test users:",
    len(test_targets)
)


# Example
for user in list(train_sequences.keys())[:3]:

    print(
        "\nUser:",
        user
    )

    print(
        "Train:",
        train_sequences[user][-10:]
    )

    print(
        "Validation:",
        val_targets[user]
    )

    print(
        "Test:",
        test_targets[user]
    )

Train users: 19412
Validation users: 19412
Test users: 19412

User: 1
Train: [1, 2, 3]
Validation: 4
Test: 5

User: 2
Train: [4, 5, 6]
Validation: 7
Test: 8

User: 3
Train: [9, 10, 11, 12, 4, 5, 13]
Validation: 14
Test: 15


In [60]:
# ============================================================
# CELL — SEQUENTIAL DATASET
# ============================================================

class SequentialDataset(Dataset):

    def __init__(
        self,
        sequences,
        max_len=50
    ):

        self.samples = []

        self.max_len = max_len


        for user, seq in sequences.items():

            # ----------------------------------------------
            # Create next-item prediction samples
            #
            # [1]       -> 2
            # [1,2]     -> 3
            # [1,2,3]   -> 4
            # ----------------------------------------------

            for i in range(
                1,
                len(seq)
            ):

                history = seq[:i]

                # Keep only last MAX_LEN items
                history = history[-max_len:]

                target = seq[i]

                self.samples.append(
                    (
                        history,
                        target
                    )
                )


    def __len__(self):

        return len(self.samples)


    def __getitem__(self, index):

        history, target = self.samples[index]

        return {
            "sequence": history,
            "target": target
        }

In [61]:
# ============================================================
# CELL — COLLATE FUNCTION
# ============================================================

def collate_fn(batch):

    sequences = [
        x["sequence"]
        for x in batch
    ]

    targets = torch.tensor(
        [
            x["target"]
            for x in batch
        ],
        dtype=torch.long
    )


    input_ids = torch.zeros(
        len(batch),
        MAX_LEN,
        dtype=torch.long
    )

    input_mask = torch.zeros(
        len(batch),
        MAX_LEN,
        dtype=torch.long
    )


    for i, seq in enumerate(sequences):

        seq = seq[-MAX_LEN:]

        length = len(seq)

        input_ids[
            i,
            -length:
        ] = torch.tensor(
            seq,
            dtype=torch.long
        )

        input_mask[
            i,
            -length:
        ] = 1


    return {

        "input_ids":
            input_ids,

        "input_mask":
            input_mask,

        "targets":
            targets
    }

In [62]:
# ============================================================
# CELL — DATALOADER
# ============================================================

train_dataset = SequentialDataset(
    train_sequences,
    max_len=MAX_LEN
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)


print(
    "Training samples:",
    len(train_dataset)
)

print(
    "Training batches:",
    len(train_loader)
)

Training samples: 109361
Training batches: 428


In [63]:
# # ============================================================
# # CELL 6 — TFRecord DATASET
# # ============================================================

# class AmazonTFRecordDataset(IterableDataset):

#     def __init__(
#         self,
#         input_file,
#         batch_size=None,
#         shuffle=False,
#         repeat=1
#     ):

#         self.input_file = input_file
#         self.batch_size = batch_size
#         self.shuffle = shuffle
#         self.repeat = repeat


#     def _parse_record(self, record):

#         # ----------------------------------------------------
#         # IMPORTANT:
#         # These names must match the TFRecord created by
#         # the original project.
#         # ----------------------------------------------------

#         feature_description = {

#             "input_ids":
#                 tf.io.FixedLenFeature(
#                     [MAX_SEQ_LENGTH + PREFERENCE_SIZE],
#                     tf.int64
#                 ),

#             "input_mask":
#                 tf.io.FixedLenFeature(
#                     [MAX_SEQ_LENGTH + PREFERENCE_SIZE],
#                     tf.int64
#                 ),

#             "segment_ids":
#                 tf.io.FixedLenFeature(
#                     [MAX_SEQ_LENGTH + PREFERENCE_SIZE],
#                     tf.int64
#                 ),

#             "label_ids":
#                 tf.io.FixedLenFeature(
#                     [MAX_SEQ_LENGTH + PREFERENCE_SIZE],
#                     tf.int64
#                 ),

#             "preference_ids":
#                 tf.io.FixedLenFeature(
#                     [PREFERENCE_SIZE],
#                     tf.int64
#                 ),

#             "preference_mask":
#                 tf.io.FixedLenFeature(
#                     [PREFERENCE_SIZE],
#                     tf.int64
#                 ),
#         }

#         example = tf.io.parse_single_example(
#             record,
#             feature_description
#         )

#         return example


#     def __iter__(self):

#         dataset = tf.data.TFRecordDataset(
#             self.input_file
#         )

#         if self.shuffle:

#             dataset = dataset.shuffle(
#                 buffer_size=10000,
#                 seed=SEED,
#                 reshuffle_each_iteration=True
#             )

#         for _ in range(self.repeat):

#             for record in dataset:

#                 example = self._parse_record(record)

#                 yield {
#                     key: value.numpy()
#                     for key, value in example.items()
#                 }

In [64]:
# # ============================================================
# # CELL 7 — COLLATE FUNCTION
# # ============================================================

# def collate_fn(batch):

#     result = {}

#     keys = batch[0].keys()

#     for key in keys:

#         values = [
#             torch.tensor(item[key])
#             for item in batch
#         ]

#         result[key] = torch.stack(values)


#     return result

In [65]:
# # ============================================================
# # CELL 8 — CREATE DATALOADERS
# # ============================================================

# train_dataset = AmazonTFRecordDataset(
#     input_file=TRAIN_FILE,
#     shuffle=True,
#     repeat=1
# )

# test_dataset = AmazonTFRecordDataset(
#     input_file=TEST_FILE,
#     shuffle=False,
#     repeat=1
# )


# train_loader = DataLoader(
#     train_dataset,
#     batch_size=BATCH_SIZE,
#     collate_fn=collate_fn,
#     num_workers=0
# )

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     collate_fn=collate_fn,
#     num_workers=0
# )


# print("DataLoaders created.")

In [66]:
# ============================================================
# CELL 9 — MULTI HEAD ATTENTION
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        hidden_size,
        num_heads,
        dropout=0.1
    ):

        super().__init__()

        assert hidden_size % num_heads == 0

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.query = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.key = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.value = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.output = nn.Linear(
            hidden_size,
            hidden_size
        )

        self.dropout = nn.Dropout(dropout)


    def forward(
        self,
        hidden_states,
        attention_mask=None
    ):

        B, L, H = hidden_states.shape

        Q = self.query(hidden_states)
        K = self.key(hidden_states)
        V = self.value(hidden_states)

        Q = Q.view(
            B,
            L,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            B,
            L,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            B,
            L,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        scores = scores / math.sqrt(self.head_dim)


        if attention_mask is not None:

            # [B, L] -> [B, 1, 1, L]

            if attention_mask.dim() == 2:

                mask = attention_mask[:, None, None, :]

                scores = scores.masked_fill(
                    mask == 0,
                    -1e9
                )

            # [B, L, L]

            elif attention_mask.dim() == 3:

                mask = attention_mask[:, None, :, :]

                scores = scores.masked_fill(
                    mask == 0,
                    -1e9
                )


        attention_probs = F.softmax(
            scores,
            dim=-1
        )

        attention_probs = self.dropout(
            attention_probs
        )

        context = torch.matmul(
            attention_probs,
            V
        )

        context = context.transpose(
            1,
            2
        ).contiguous()

        context = context.view(
            B,
            L,
            H
        )

        output = self.output(context)

        return output, attention_probs

In [67]:
# ============================================================
# CELL 10 — TRANSFORMER BLOCK
# ============================================================

class TransformerBlock(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.attention = MultiHeadAttention(
            config.hidden_size,
            config.num_attention_heads,
            config.attention_probs_dropout_prob
        )

        self.norm1 = nn.LayerNorm(
            config.hidden_size
        )

        self.ffn = nn.Sequential(

            nn.Linear(
                config.hidden_size,
                config.intermediate_size
            ),

            nn.GELU(),

            nn.Linear(
                config.intermediate_size,
                config.hidden_size
            )
        )

        self.dropout = nn.Dropout(
            config.hidden_dropout_prob
        )

        self.norm2 = nn.LayerNorm(
            config.hidden_size
        )


    def forward(
        self,
        x,
        attention_mask=None
    ):

        attention_output, attention_probs = (
            self.attention(
                x,
                attention_mask
            )
        )

        x = self.norm1(
            x + self.dropout(attention_output)
        )

        ff = self.ffn(x)

        x = self.norm2(
            x + self.dropout(ff)
        )

        return x, attention_probs

In [68]:
# ============================================================
# CELL 11 — SHARED EMBEDDING DECODER
# ============================================================

class SharedEmbeddingDecoder(nn.Module):

    def __init__(
        self,
        config,
        embedding_layer
    ):

        super().__init__()

        self.embedding_layer = embedding_layer

        self.dense = nn.Linear(
            config.hidden_size,
            config.hidden_size
        )

        self.norm = nn.LayerNorm(
            config.hidden_size
        )


    def forward(self, hidden_states):

        x = self.dense(hidden_states)

        x = F.gelu(x)

        x = self.norm(x)

        # Weight tying
        logits = torch.matmul(
            x,
            self.embedding_layer.weight.T
        )

        return F.log_softmax(
            logits,
            dim=-1
        )

In [69]:
# ============================================================
# CELL 12 — PREFERENCE SEPARATION
# ============================================================

class PreferenceSeparation(nn.Module):

    def __init__(
        self,
        hidden_size,
        preference_size
    ):

        super().__init__()

        self.preference_size = preference_size

        self.projection = nn.Linear(
            hidden_size,
            hidden_size
        )


    def forward(self, preferences):

        # preferences:
        # [B, P, H]

        projected = self.projection(
            preferences
        )

        projected = F.normalize(
            projected,
            p=2,
            dim=-1
        )

        return projected

In [70]:
# ============================================================
# CELL 13 — MAIN MODEL
# ============================================================

class BertPreferenceModel(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.config = config

        self.preference_size = config.preference_size

        # ----------------------------------------------------
        # Embeddings
        # ----------------------------------------------------

        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size
        )

        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings
            + config.preference_size,
            config.hidden_size
        )

        self.token_type_embeddings = nn.Embedding(
            config.type_vocab_size,
            config.hidden_size
        )

        self.embedding_norm = nn.LayerNorm(
            config.hidden_size
        )

        self.embedding_dropout = nn.Dropout(
            config.hidden_dropout_prob
        )

        # ----------------------------------------------------
        # Transformer
        # ----------------------------------------------------

        self.layers = nn.ModuleList([

            TransformerBlock(config)

            for _ in range(
                config.num_hidden_layers
            )

        ])

        # ----------------------------------------------------
        # Preference identifier
        # ----------------------------------------------------

        self.identifier = MultiHeadAttention(
            config.hidden_size,
            config.num_attention_heads,
            config.attention_probs_dropout_prob
        )

        # ----------------------------------------------------
        # Preference separation
        # ----------------------------------------------------

        self.preference_separation = (
            PreferenceSeparation(
                config.hidden_size,
                config.preference_size
            )
        )

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        self.decoder = SharedEmbeddingDecoder(
            config,
            self.word_embeddings
        )

        # ----------------------------------------------------
        # Initialize
        # ----------------------------------------------------

        self._initialize_weights()


    # ========================================================
    # Weight initialization
    # ========================================================

    def _initialize_weights(self):

        nn.init.normal_(
            self.word_embeddings.weight,
            mean=0.0,
            std=0.02
        )

        nn.init.normal_(
            self.position_embeddings.weight,
            mean=0.0,
            std=0.02
        )

        nn.init.normal_(
            self.token_type_embeddings.weight,
            mean=0.0,
            std=0.02
        )


    # ========================================================
    # Embedding
    # ========================================================

    def embed(
        self,
        input_ids,
        segment_ids
    ):

        B, L = input_ids.shape

        positions = torch.arange(
            L,
            device=input_ids.device
        )

        positions = positions.unsqueeze(0).expand(
            B,
            L
        )

        x = (
            self.word_embeddings(input_ids)

            + self.position_embeddings(positions)

            + self.token_type_embeddings(segment_ids)
        )

        x = self.embedding_norm(x)

        x = self.embedding_dropout(x)

        return x


    # ========================================================
    # Encoder
    # ========================================================

    def encode(
        self,
        input_ids,
        input_mask,
        segment_ids
    ):

        x = self.embed(
            input_ids,
            segment_ids
        )

        attentions = []

        for layer in self.layers:

            x, attention = layer(
                x,
                input_mask
            )

            attentions.append(
                attention
            )

        return x, attentions


    # ========================================================
    # Preference representation
    # ========================================================

    def identify_preferences(
        self,
        hidden_states
    ):

        # Last preference_size positions
        # are used as preference representations.

        preferences = hidden_states[
            :,
            -self.preference_size:,
            :
        ]

        preferences = self.preference_separation(
            preferences
        )

        return preferences


    # ========================================================
    # Prediction loss
    # ========================================================

    def prediction_loss(
        self,
        hidden_states,
        label_ids,
        label_mask=None
    ):

        log_probs = self.decoder(
            hidden_states
        )

        B, L, V = log_probs.shape

        loss = F.nll_loss(
            log_probs.reshape(-1, V),
            label_ids.reshape(-1),
            reduction="none"
        )

        loss = loss.reshape(
            B,
            L
        )

        if label_mask is not None:

            loss = (
                loss
                * label_mask.float()
            )

            denominator = (
                label_mask.float()
                .sum()
                .clamp_min(1.0)
            )

            loss = (
                loss.sum()
                / denominator
            )

        else:

            loss = loss.mean()

        return loss, log_probs


    # ========================================================
    # Coverage Loss
    # ========================================================

    def coverage_loss(
        self,
        attentions,
        coverage_mask
    ):

        if len(attentions) == 0:

            return torch.tensor(
                0.0,
                device=coverage_mask.device
            )

        # ----------------------------------------------------
        # Last Transformer attention
        # ----------------------------------------------------

        attention = attentions[-1]

        # [B, heads, L, L]
        attention = attention.mean(
            dim=1
        )

        # ----------------------------------------------------
        # Sequence part
        # ----------------------------------------------------

        item_attention = attention[
            :,
            :,
            :MAX_SEQ_LENGTH
        ]

        coverage = torch.zeros(
            item_attention.size(0),
            item_attention.size(-1),
            device=item_attention.device
        )

        total_loss = torch.zeros(
            item_attention.size(0),
            device=item_attention.device
        )

        # ----------------------------------------------------
        # Coverage over preference steps
        # ----------------------------------------------------

        steps = min(
            item_attention.size(1),
            self.preference_size
        )

        for t in range(steps):

            current = item_attention[
                :,
                t,
                :
            ]

            overlap = torch.minimum(
                current,
                coverage
            )

            total_loss += overlap.sum(
                dim=-1
            )

            coverage = (
                coverage
                + current
            )

        # ----------------------------------------------------
        # Coverage mask
        # ----------------------------------------------------

        mask = coverage_mask[
            :,
            :item_attention.size(-1)
        ].float()

        denominator = (
            mask.sum(
                dim=-1
            ).clamp_min(1.0)
        )

        if mask.shape == coverage.shape:

            masked_coverage = (
                coverage * mask
            )

        else:

            masked_coverage = coverage

        result = (
            total_loss
            / denominator
        )

        return result.mean()


    # ========================================================
    # SSL Forward
    # ========================================================

    def ssl_forward(
        self,
        input_ids,
        input_mask,
        segment_ids,
        label_ids,
        label_mask,
        preference_ids=None,
        preference_mask=None
    ):

        hidden_states, attentions = self.encode(
            input_ids,
            input_mask,
            segment_ids
        )

        preferences = (
            self.identify_preferences(
                hidden_states
            )
        )

        # ----------------------------------------------------
        # Common representation loss
        # ----------------------------------------------------

        common_loss = (
            preferences.mean(
                dim=1
            )
            .pow(2)
            .mean()
        )

        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        pred_loss, log_probs = (
            self.prediction_loss(
                hidden_states,
                label_ids,
                label_mask
            )
        )

        ssl_loss = (
            SSL_COMMON_WEIGHT
            * common_loss
            +
            SSL_NEW_PRED_WEIGHT
            * pred_loss
        )

        return {

            "loss_ssl": ssl_loss,

            "common_loss": common_loss,

            "pred_loss": pred_loss,

            "log_probs": log_probs
        }


    # ========================================================
    # Fine-tuning
    # ========================================================

    def finetune_loss(
        self,
        input_ids,
        input_mask,
        segment_ids,
        label_ids,
        label_mask,
        preference_ids=None,
        preference_mask=None
    ):

        hidden_states, attentions = self.encode(
            input_ids,
            input_mask,
            segment_ids
        )

        preferences = (
            self.identify_preferences(
                hidden_states
            )
        )

        pred_loss, log_probs = (
            self.prediction_loss(
                hidden_states,
                label_ids,
                label_mask
            )
        )

        # ----------------------------------------------------
        # Coverage mask
        # ----------------------------------------------------

        if input_mask is not None:

            coverage_mask = input_mask[
                :,
                :MAX_SEQ_LENGTH
            ]

        else:

            coverage_mask = torch.ones(
                input_ids.size(0),
                MAX_SEQ_LENGTH,
                device=input_ids.device
            )

        cov_loss = self.coverage_loss(
            attentions,
            coverage_mask
        )

        total_loss = (
            pred_loss
            +
            COVERAGE_WEIGHT * cov_loss
        )

        return {

            "loss": total_loss,

            "pred_loss": pred_loss,

            "coverage_loss": cov_loss,

            "log_probs": log_probs,

            "preferences": preferences
        }


    # ========================================================
    # Forward — Toys next-item prediction
    # ========================================================

    def forward(
        self,
        input_ids,
        input_mask
    ):

        # ----------------------------------------------------
        # Segment IDs
        #
        # The new toys_pro.txt dataset does not contain
        # segment IDs, so we use segment 0 for every item.
        # ----------------------------------------------------

        segment_ids = torch.zeros_like(
            input_ids
        )

        # ----------------------------------------------------
        # Encoder
        # ----------------------------------------------------

        hidden_states, attentions = self.encode(
            input_ids=input_ids,
            input_mask=input_mask,
            segment_ids=segment_ids
        )

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        logits = self.decoder(
            hidden_states
        )

        return {

            "logits": logits,

            "hidden_states": hidden_states,

            "attentions": attentions
        }


# ============================================================
# IMPORTANT — Vocabulary size for toys_pro.txt
# ============================================================

# The new dataset uses:
#
#     0              -> padding
#     1..NUM_ITEMS   -> real items
#
# Therefore the model vocabulary must be:
#
#     NUM_ITEMS + 1

config.vocab_size = NUM_ITEMS + 1


# ============================================================
# Create model
# ============================================================

model = BertPreferenceModel(
    config
).to(device)


# ============================================================
# Model information
# ============================================================

print(
    model.__class__.__name__
)

num_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    "Trainable parameters:",
    f"{num_parameters:,}"
)

print(
    "NUM_ITEMS:",
    NUM_ITEMS
)

print(
    "Model vocabulary size:",
    config.vocab_size
)

BertPreferenceModel
Trainable parameters: 893,312
NUM_ITEMS: 11924
Model vocabulary size: 11925


In [71]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
input_mask = batch["input_mask"].to(device)

with torch.no_grad():
    output = model(
        input_ids=input_ids,
        input_mask=input_mask
    )

print(output["logits"].shape)

torch.Size([256, 50, 11925])


In [72]:
# ============================================================
# CELL 14 — OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print(optimizer)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0003
    maximize: False
    weight_decay: 0.01
)


In [73]:
# # ============================================================
# # CELL 15 — COUNT TRAINING RECORDS
# # ============================================================

# def count_tfrecord_records(path):

#     dataset = tf.data.TFRecordDataset(path)

#     count = 0

#     for _ in dataset:
#         count += 1

#     return count


# print("Counting TFRecord samples...")

# NUM_TRAIN_RECORDS = count_tfrecord_records(
#     TRAIN_FILE
# )

# print(
#     "Training records:",
#     NUM_TRAIN_RECORDS
# )


# STEPS_PER_EPOCH = math.ceil(
#     NUM_TRAIN_RECORDS / BATCH_SIZE
# )

# TOTAL_EPOCHS = (
#     EPOCHS_PRETRAIN
#     +
#     EPOCHS_FINETUNE
# )

# TOTAL_STEPS = (
#     STEPS_PER_EPOCH
#     *
#     TOTAL_EPOCHS
# )

# WARMUP_STEPS = max(
#     1,
#     int(TOTAL_STEPS * WARMUP_RATIO)
# )


# print("Steps / epoch :", STEPS_PER_EPOCH)
# print("Total epochs  :", TOTAL_EPOCHS)
# print("Total steps   :", TOTAL_STEPS)
# print("Warmup steps  :", WARMUP_STEPS)

In [74]:
# ============================================================
# CELL 16 — LINEAR WARMUP + DECAY
# ============================================================

TOTAL_TRAIN_STEPS = (
    EPOCHS_PRETRAIN + EPOCHS_FINETUNE
) * len(train_loader)

WARMUP_STEPS = int(0.1 * TOTAL_TRAIN_STEPS)


def lr_lambda(current_step):

    if current_step < WARMUP_STEPS:
        return float(current_step + 1) / float(
            max(1, WARMUP_STEPS)
        )

    return max(
        0.0,
        float(TOTAL_TRAIN_STEPS - current_step)
        / float(
            max(1, TOTAL_TRAIN_STEPS - WARMUP_STEPS)
        )
    )


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)

print("Total steps:", TOTAL_TRAIN_STEPS)
print("Warmup steps:", WARMUP_STEPS)
print("Scheduler ready.")

Total steps: 17120
Warmup steps: 1712
Scheduler ready.


In [75]:
# ============================================================
# CELL 17 — MOVE BATCH TO GPU
# ============================================================

def move_batch_to_device(batch):

    return {
        key: value.to(
            device,
            non_blocking=True
        )

        for key, value in batch.items()
    }

In [76]:
# ============================================================
# PRETRAIN ONE EPOCH
# ============================================================

def pretrain_one_epoch(
    model,
    loader,
    optimizer,
    scheduler=None,
    epoch=1
):

    model.train()

    total_loss = 0.0
    total_common_loss = 0.0
    total_pred_loss = 0.0

    num_batches = 0

    progress = tqdm(
        loader,
        desc=f"Pretrain Epoch {epoch}",
        leave=True
    )

    for batch in progress:

        # ----------------------------------------------------
        # Data
        # ----------------------------------------------------

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        input_mask = batch["input_mask"].to(
            device,
            non_blocking=True
        )

        targets = batch["targets"].to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Segment IDs
        #
        # toys_pro.txt does not contain segment IDs.
        # Therefore all tokens belong to segment 0.
        # ----------------------------------------------------

        segment_ids = torch.zeros_like(
            input_ids
        )

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------
        #
        # The current Dataset gives:
        #
        # input_ids -> user's history
        # targets   -> next item
        #
        # We use the last valid position of the sequence
        # as the prediction position.
        # ----------------------------------------------------

        hidden_states, attentions = model.encode(
            input_ids=input_ids,
            input_mask=input_mask,
            segment_ids=segment_ids
        )

        # ----------------------------------------------------
        # Preference representations
        # ----------------------------------------------------

        preferences = model.identify_preferences(
            hidden_states
        )

        # ----------------------------------------------------
        # Common loss
        # ----------------------------------------------------

        common_loss = (
            preferences.mean(dim=1)
            .pow(2)
            .mean()
        )

        # ----------------------------------------------------
        # Decoder
        # ----------------------------------------------------

        log_probs = model.decoder(
            hidden_states
        )

        # ----------------------------------------------------
        # Last valid token position
        #
        # IMPORTANT:
        # The current collate_fn uses LEFT PADDING.
        #
        # Therefore the last valid position is NOT:
        #
        #     length - 1
        #
        # It is the last column:
        #
        #     MAX_LEN - 1
        #
        # ----------------------------------------------------

        prediction_position = (
            input_mask.sum(dim=1) > 0
        )

        # Safety check
        if not prediction_position.any():

            continue

        # Since sequences are left padded, the last
        # non-padding position is MAX_LEN - 1.
        last_position = (
            input_ids.size(1) - 1
        )

        prediction_log_probs = log_probs[
            :,
            last_position,
            :
        ]

        # ----------------------------------------------------
        # Next-item prediction loss
        # ----------------------------------------------------

        pred_loss = F.nll_loss(
            prediction_log_probs,
            targets,
            reduction="mean"
        )

        # ----------------------------------------------------
        # SSL total loss
        # ----------------------------------------------------

        ssl_loss = (
            SSL_COMMON_WEIGHT * common_loss
            +
            SSL_NEW_PRED_WEIGHT * pred_loss
        )

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        ssl_loss.backward()

        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # ----------------------------------------------------
        # Optimizer
        # ----------------------------------------------------

        optimizer.step()

        # ----------------------------------------------------
        # Scheduler
        # ----------------------------------------------------

        if scheduler is not None:

            scheduler.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        total_loss += (
            ssl_loss.detach().item()
        )

        total_common_loss += (
            common_loss.detach().item()
        )

        total_pred_loss += (
            pred_loss.detach().item()
        )

        num_batches += 1

        # ----------------------------------------------------
        # Progress bar
        # ----------------------------------------------------

        progress.set_postfix(
            loss=f"{ssl_loss.item():.4f}"
        )

    # ========================================================
    # Epoch statistics
    # ========================================================

    if num_batches == 0:

        return {
            "loss": 0.0,
            "common_loss": 0.0,
            "pred_loss": 0.0
        }

    avg_loss = (
        total_loss / num_batches
    )

    avg_common_loss = (
        total_common_loss
        / num_batches
    )

    avg_pred_loss = (
        total_pred_loss
        / num_batches
    )

    return {

        "loss": avg_loss,

        "common_loss": avg_common_loss,

        "pred_loss": avg_pred_loss
    }

In [77]:
# ============================================================
# CELL 19 — CHECKPOINT FUNCTIONS
# ============================================================

def save_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    epoch,
    loss,
    stage
):

    checkpoint = {

        "epoch": epoch,

        "loss": loss,

        "stage": stage,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "config":
            vars(config),

        "seed":
            SEED
    }

    torch.save(
        checkpoint,
        path
    )


    print(
        f"Checkpoint saved: {path}"
    )


def load_checkpoint(
    path,
    model,
    optimizer=None,
    scheduler=None
):

    checkpoint = torch.load(
        path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )


    if optimizer is not None:

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )


    if scheduler is not None:

        scheduler.load_state_dict(
            checkpoint["scheduler_state_dict"]
        )


    print(
        "Checkpoint loaded:",
        path
    )

    print(
        "Stage:",
        checkpoint.get("stage")
    )

    print(
        "Epoch:",
        checkpoint.get("epoch")
    )

    print(
        "Loss:",
        checkpoint.get("loss")
    )

    return checkpoint

In [78]:
# ============================================================
# CELL 20 — PRETRAIN
# ============================================================

best_pretrain_loss = float("inf")


for epoch in range(
    1,
    EPOCHS_PRETRAIN + 1
):

    metrics = pretrain_one_epoch(

        model=model,

        loader=train_loader,

        optimizer=optimizer,

        scheduler=scheduler,

        epoch=epoch
    )


    print(
        f"\n[PRETRAIN] "
        f"Epoch {epoch}/{EPOCHS_PRETRAIN} | "
        f"Loss: {metrics['loss']:.6f} | "
        f"Common: {metrics['common_loss']:.6f} | "
        f"Pred: {metrics['pred_loss']:.6f}"
    )


    if metrics["loss"] < best_pretrain_loss:

        best_pretrain_loss = metrics["loss"]

        save_checkpoint(

            BEST_PRETRAIN_CHECKPOINT,

            model,

            optimizer,

            scheduler,

            epoch,

            metrics["loss"],

            "pretrain"
        )

Pretrain Epoch 1:   0%|          | 0/428 [00:00<?, ?it/s]


[PRETRAIN] Epoch 1/20 | Loss: 0.968099 | Common: 0.002989 | Pred: 9.382129
Checkpoint saved: /content/checkpoint/toys_pytorch/best_pretrain.pt


Pretrain Epoch 2:   0%|          | 0/428 [00:00<?, ?it/s]


[PRETRAIN] Epoch 2/20 | Loss: 0.925922 | Common: 0.000260 | Pred: 9.233268
Checkpoint saved: /content/checkpoint/toys_pytorch/best_pretrain.pt


Pretrain Epoch 3:   0%|          | 0/428 [00:00<?, ?it/s]

...


[PRETRAIN] Epoch 18/20 | Loss: 0.632871 | Common: 0.000015 | Pred: 6.327214
Checkpoint saved: /content/checkpoint/toys_pytorch/best_pretrain.pt


Pretrain Epoch 19:   0%|          | 0/428 [00:00<?, ?it/s]


[PRETRAIN] Epoch 19/20 | Loss: 0.623616 | Common: 0.000014 | Pred: 6.234712
Checkpoint saved: /content/checkpoint/toys_pytorch/best_pretrain.pt


Pretrain Epoch 20:   0%|          | 0/428 [00:00<?, ?it/s]


[PRETRAIN] Epoch 20/20 | Loss: 0.614773 | Common: 0.000014 | Pred: 6.146362
Checkpoint saved: /content/checkpoint/toys_pytorch/best_pretrain.pt


In [79]:
print(type(model))
print(model.__class__.__name__)
print(model.forward)

<class '__main__.BertPreferenceModel'>
BertPreferenceModel
<bound method BertPreferenceModel.forward of BertPreferenceModel(
  (word_embeddings): Embedding(11925, 64)
  (position_embeddings): Embedding(61, 64)
  (token_type_embeddings): Embedding(16, 64)
  (embedding_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (embedding_dropout): Dropout(p=0.2, inplace=False)
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (attention): MultiHeadAttention(
        (query): Linear(in_features=64, out_features=64, bias=True)
        (key): Linear(in_features=64, out_features=64, bias=True)
        (value): Linear(in_features=64, out_features=64, bias=True)
        (output): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')

In [80]:
import inspect

print(inspect.signature(model.forward))

(input_ids, input_mask)


In [81]:
# ============================================================
# FINETUNE — ONE EPOCH
# ============================================================

def finetune_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    epoch
):

    model.train()

    total_loss = 0.0
    total_pred_loss = 0.0
    total_coverage_loss = 0.0

    total_samples = 0

    progress = tqdm(
        loader,
        desc=f"Finetune Epoch {epoch}",
        leave=True
    )

    for batch in progress:

        # ----------------------------------------------------
        # Move data to device
        # ----------------------------------------------------

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        input_mask = batch["input_mask"].to(
            device,
            non_blocking=True
        )

        targets = batch["targets"].to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Segment IDs
        #
        # toys_pro.txt does not have segment IDs.
        # Everything belongs to segment 0.
        # ----------------------------------------------------

        segment_ids = torch.zeros_like(
            input_ids
        )

        # ----------------------------------------------------
        # Label IDs
        #
        # The current Dataset contains one next-item target
        # per sequence.
        #
        # We create a label tensor with the same shape as
        # input_ids and put the target at the LAST position.
        # ----------------------------------------------------

        label_ids = torch.zeros_like(
            input_ids
        )

        label_mask = torch.zeros_like(
            input_mask
        )

        # Because the sequences are LEFT-PADDED,
        # the last valid position is always:
        #
        # input_ids.size(1) - 1
        #
        last_position = (
            input_ids.size(1) - 1
        )

        label_ids[:, last_position] = targets

        label_mask[:, last_position] = (
            (targets != 0).long()
        )

        # ----------------------------------------------------
        # Forward / Fine-tuning loss
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        result = model.finetune_loss(

            input_ids=input_ids,

            input_mask=input_mask,

            segment_ids=segment_ids,

            label_ids=label_ids,

            label_mask=label_mask
        )

        # ----------------------------------------------------
        # Get losses
        # ----------------------------------------------------

        loss = result["loss"]

        pred_loss = result["pred_loss"]

        coverage_loss = result["coverage_loss"]

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        loss.backward()

        # ----------------------------------------------------
        # Gradient clipping
        # ----------------------------------------------------

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # ----------------------------------------------------
        # Optimizer
        # ----------------------------------------------------

        optimizer.step()

        # ----------------------------------------------------
        # Scheduler
        # ----------------------------------------------------

        if scheduler is not None:

            scheduler.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        batch_size = input_ids.size(0)

        total_loss += (
            loss.detach().item()
            * batch_size
        )

        total_pred_loss += (
            pred_loss.detach().item()
            * batch_size
        )

        total_coverage_loss += (
            coverage_loss.detach().item()
            * batch_size
        )

        total_samples += batch_size

        # ----------------------------------------------------
        # Running averages
        # ----------------------------------------------------

        avg_loss = (
            total_loss
            / max(1, total_samples)
        )

        avg_pred_loss = (
            total_pred_loss
            / max(1, total_samples)
        )

        avg_coverage_loss = (
            total_coverage_loss
            / max(1, total_samples)
        )

        # ----------------------------------------------------
        # Progress bar
        # ----------------------------------------------------

        progress.set_postfix({

            "loss": f"{avg_loss:.4f}",

            "pred": f"{avg_pred_loss:.4f}",

            "cov": f"{avg_coverage_loss:.4f}"

        })

    # ========================================================
    # Epoch statistics
    # ========================================================

    epoch_loss = (
        total_loss
        / max(1, total_samples)
    )

    epoch_pred_loss = (
        total_pred_loss
        / max(1, total_samples)
    )

    epoch_coverage_loss = (
        total_coverage_loss
        / max(1, total_samples)
    )

    return {

        "loss": epoch_loss,

        "pred_loss": epoch_pred_loss,

        "coverage_loss": epoch_coverage_loss
    }

In [82]:
# ============================================================
# CELL 22 — FINE-TUNING
# ============================================================

best_finetune_loss = float("inf")


for epoch in range(
    1,
    EPOCHS_FINETUNE + 1
):

    metrics = finetune_one_epoch(

        model=model,

        loader=train_loader,

        optimizer=optimizer,

        scheduler=scheduler,

        epoch=epoch
    )


    print(
        f"\n[FINETUNE] "
        f"Epoch {epoch}/{EPOCHS_FINETUNE} | "
        f"Loss: {metrics['loss']:.6f} | "
        f"Pred: {metrics['pred_loss']:.6f} | "
        f"Coverage: {metrics['coverage_loss']:.6f}"
    )


    # --------------------------------------------------------
    # Save every epoch
    # --------------------------------------------------------

    save_checkpoint(

        LAST_CHECKPOINT,

        model,

        optimizer,

        scheduler,

        epoch,

        metrics["loss"],

        "finetune"
    )


    # --------------------------------------------------------
    # Best model
    # --------------------------------------------------------

    if metrics["loss"] < best_finetune_loss:

        best_finetune_loss = metrics["loss"]

        save_checkpoint(

            BEST_CHECKPOINT,

            model,

            optimizer,

            scheduler,

            epoch,

            metrics["loss"],

            "finetune"
        )

Finetune Epoch 1:   0%|          | 0/428 [00:00<?, ?it/s]


[FINETUNE] Epoch 1/20 | Loss: 9.357566 | Pred: 6.155909 | Coverage: 3.201658
Checkpoint saved: /content/checkpoint/toys_pytorch/last_model.pt
Checkpoint saved: /content/checkpoint/toys_pytorch/best_model.pt


Finetune Epoch 2:   0%|          | 0/428 [00:00<?, ?it/s]


[FINETUNE] Epoch 2/20 | Loss: 9.093665 | Pred: 5.994993 | Coverage: 3.098672
Checkpoint saved: /content/checkpoint/toys_pytorch/last_model.pt
Checkpoint saved: /content/checkpoint/toys_pytorch/best_model.pt


Finetune Epoch 3:   0%|          | 0/428 [00:00<?, ?it/s]

...


[FINETUNE] Epoch 18/20 | Loss: 8.406372 | Pred: 5.326045 | Coverage: 3.080328
Checkpoint saved: /content/checkpoint/toys_pytorch/last_model.pt
Checkpoint saved: /content/checkpoint/toys_pytorch/best_model.pt


Finetune Epoch 19:   0%|          | 0/428 [00:00<?, ?it/s]


[FINETUNE] Epoch 19/20 | Loss: 8.391866 | Pred: 5.314066 | Coverage: 3.077801
Checkpoint saved: /content/checkpoint/toys_pytorch/last_model.pt
Checkpoint saved: /content/checkpoint/toys_pytorch/best_model.pt


Finetune Epoch 20:   0%|          | 0/428 [00:00<?, ?it/s]


[FINETUNE] Epoch 20/20 | Loss: 8.384555 | Pred: 5.307318 | Coverage: 3.077237
Checkpoint saved: /content/checkpoint/toys_pytorch/last_model.pt
Checkpoint saved: /content/checkpoint/toys_pytorch/best_model.pt


In [83]:
# ============================================================
# CELL 23 — LOAD BEST MODEL
# ============================================================

best_checkpoint = load_checkpoint(
    BEST_CHECKPOINT,
    model
)

model.eval()

print(
    "\nBest fine-tuning loss:",
    best_checkpoint["loss"]
)

Checkpoint loaded: /content/checkpoint/toys_pytorch/best_model.pt
Stage: finetune
Epoch: 20
Loss: 8.384554941067474

Best fine-tuning loss: 8.384554941067474


In [84]:
# ============================================================
# CELL 24 — LOAD HISTORY
# ============================================================

def load_history(path):

    history = {}

    if not os.path.exists(path):

        print(
            "History file not found."
        )

        return history


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            parts = line.split()

            if len(parts) < 2:
                continue

            user_id = int(parts[0])

            items = [
                int(x)
                for x in parts[1:]
            ]

            history[user_id] = items


    return history


history = load_history(
    HISTORY_FILE
)

print(
    "History users:",
    len(history)
)

History file not found.
History users: 0


In [85]:
# ============================================================
# TEST DATASET
# ============================================================

class TestSequentialDataset(Dataset):

    def __init__(
        self,
        train_sequences,
        test_targets,
        max_len=50
    ):

        self.samples = []

        self.max_len = max_len

        for user in train_sequences:

            if user not in test_targets:
                continue

            history = train_sequences[user]

            target = test_targets[user]

            if len(history) == 0:
                continue

            self.samples.append(
                (
                    history,
                    target
                )
            )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, index):

        history, target = self.samples[index]

        return {
            "sequence": history,
            "target": target
        }


# ============================================================
# TEST COLLATE
# ============================================================

def test_collate_fn(batch):

    sequences = [
        x["sequence"]
        for x in batch
    ]

    targets = torch.tensor(
        [
            x["target"]
            for x in batch
        ],
        dtype=torch.long
    )

    input_ids = torch.zeros(
        len(batch),
        MAX_LEN,
        dtype=torch.long
    )

    input_mask = torch.zeros(
        len(batch),
        MAX_LEN,
        dtype=torch.long
    )

    for i, seq in enumerate(sequences):

        seq = seq[-MAX_LEN:]

        length = len(seq)

        input_ids[
            i,
            -length:
        ] = torch.tensor(
            seq,
            dtype=torch.long
        )

        input_mask[
            i,
            -length:
        ] = 1

    return {

        "input_ids": input_ids,

        "input_mask": input_mask,

        "targets": targets
    }


# ============================================================
# CREATE TEST DATASET
# ============================================================

test_dataset = TestSequentialDataset(

    train_sequences=train_sequences,

    test_targets=test_targets,

    max_len=MAX_LEN
)


# ============================================================
# CREATE TEST LOADER
# ============================================================

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available(),

    collate_fn=test_collate_fn
)


# ============================================================
# INFORMATION
# ============================================================

print(
    "Test samples:",
    len(test_dataset)
)

print(
    "Test batches:",
    len(test_loader)
)

Test samples: 19412
Test batches: 76


In [86]:
# ============================================================
# EVALUATION — TOYS
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    test_loader,
    top_k_list=(5, 10, 20),
    num_negatives=100
):

    model.eval()

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    result = {}

    for k in top_k_list:

        result[f"Recall@{k}"] = 0.0
        result[f"MRR@{k}"] = 0.0
        result[f"NDCG@{k}"] = 0.0

    total = 0

    # --------------------------------------------------------
    # All items
    # --------------------------------------------------------

    all_items = torch.arange(
        1,
        NUM_ITEMS + 1,
        device=device,
        dtype=torch.long
    )

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    progress = tqdm(
        test_loader,
        desc="Evaluation"
    )

    for batch in progress:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        input_mask = batch["input_mask"].to(
            device,
            non_blocking=True
        )

        targets = batch["targets"].to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        outputs = model(
            input_ids=input_ids,
            input_mask=input_mask
        )

        if isinstance(outputs, dict):

            if "logits" in outputs:

                logits = outputs["logits"]

            elif "prediction_logits" in outputs:

                logits = outputs["prediction_logits"]

            else:

                raise KeyError(
                    "Model output must contain "
                    "'logits' or 'prediction_logits'."
                )

        else:

            logits = outputs

        # ----------------------------------------------------
        # Last valid position
        #
        # Dataset uses LEFT PADDING.
        # Therefore the last item is at MAX_LEN - 1.
        # ----------------------------------------------------

        target_position = (
            input_ids.size(1) - 1
        )

        batch_size = input_ids.size(0)

        # ----------------------------------------------------
        # Process each user
        # ----------------------------------------------------

        for i in range(batch_size):

            target = targets[i]

            if target.item() == 0:
                continue

            # ------------------------------------------------
            # User history
            # ------------------------------------------------

            history = input_ids[i]

            seen = history[
                input_mask[i].bool()
            ]

            seen = set(
                seen.detach()
                .cpu()
                .tolist()
            )

            # ------------------------------------------------
            # Candidate items
            # ------------------------------------------------

            candidate_mask = torch.ones(
                NUM_ITEMS,
                dtype=torch.bool,
                device=device
            )

            # Item IDs are 1..NUM_ITEMS
            # all_items contains exactly those IDs.

            for item in seen:

                if 1 <= item <= NUM_ITEMS:

                    candidate_mask[
                        item - 1
                    ] = False

            # Do not include target initially
            if 1 <= target.item() <= NUM_ITEMS:

                candidate_mask[
                    target.item() - 1
                ] = False

            negative_candidates = all_items[
                candidate_mask
            ]

            # ------------------------------------------------
            # Sample negatives
            # ------------------------------------------------

            if len(negative_candidates) > num_negatives:

                random_indices = torch.randperm(
                    len(negative_candidates),
                    device=device
                )[:num_negatives]

                negatives = (
                    negative_candidates[
                        random_indices
                    ]
                )

            else:

                negatives = negative_candidates

            # ------------------------------------------------
            # Target + negatives
            # ------------------------------------------------

            candidates = torch.cat(
                [
                    target.view(1),
                    negatives
                ]
            )

            # ------------------------------------------------
            # Scores
            # ------------------------------------------------

            scores = logits[
                i,
                target_position,
                candidates
            ]

            # ------------------------------------------------
            # Ranking
            # ------------------------------------------------

            ranking = torch.argsort(
                scores,
                descending=True
            )

            ranked_items = candidates[
                ranking
            ]

            target_positions = (
                ranked_items == target
            ).nonzero(
                as_tuple=False
            )

            if target_positions.numel() == 0:

                continue

            rank = (
                target_positions[0].item()
                + 1
            )

            # ------------------------------------------------
            # Metrics
            # ------------------------------------------------

            for k in top_k_list:

                if rank <= k:

                    result[
                        f"Recall@{k}"
                    ] += 1.0

                    result[
                        f"MRR@{k}"
                    ] += (
                        1.0 / rank
                    )

                    result[
                        f"NDCG@{k}"
                    ] += (
                        1.0
                        / math.log2(
                            rank + 1
                        )
                    )

            total += 1

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if total > 0:

            progress.set_postfix({

                "R@5": (
                    result["Recall@5"]
                    / total
                ),

                "R@10": (
                    result["Recall@10"]
                    / total
                ),

                "R@20": (
                    result["Recall@20"]
                    / total
                )

            })

    # --------------------------------------------------------
    # Average
    # --------------------------------------------------------

    if total == 0:

        return result

    for key in result:

        result[key] /= total

    return result

In [87]:
# ============================================================
# RUN EVALUATION
# ============================================================

results = evaluate(
    model=model,
    test_loader=test_loader,
    top_k_list=(5, 10, 20),
    num_negatives=100
)

print("\n==============================")
print("TEST RESULTS")
print("==============================")

for key, value in results.items():

    print(
        f"{key}: {value:.6f}"
    )

Evaluation:   0%|          | 0/76 [00:00<?, ?it/s]


TEST RESULTS
Recall@5: 0.270348
MRR@5: 0.174277
NDCG@5: 0.198111
Recall@10: 0.359623
MRR@10: 0.186029
NDCG@10: 0.226816
Recall@20: 0.481146
MRR@20: 0.194292
NDCG@20: 0.257328
